In [1]:
from spleeter.separator import Separator

In [2]:
import os
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
import tensorflow as tf
#libraries for building the model
from tensorflow.keras.layers import BatchNormalization, Conv2D,MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping




# Converting all the audio files into the .wav format
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment # will convert the format

from tensorflow.image import resize

import subprocess

import sys
from spleeter.separator import Separator

c:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\hindi_music\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## loading the model

In [3]:
model = tf.keras.models.load_model("indian_music_classifier_v2.keras")
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 150, 150, 32)      320       
                                                                 
 batch_normalization (BatchN  (None, 150, 150, 32)     128       
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 148, 148, 32)      9248      
                                                                 
 batch_normalization_1 (Batc  (None, 148, 148, 32)     128       
 hNormalization)                                                 
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 32)       0         
 )                                                               
                                                        

In [7]:
import json
with open('training_hist_v2.json', 'r') as json_file:
    training_history = json.load(json_file)

training_history.keys() #it is a dictionary now

dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy', 'lr'])

## Singe Audio preprocessing

In [37]:
# wav conversion done by spleeter automatically

# masking - spleeter separation

# chunking of the other version, saving mel and lables(not required) in X_test and Y_test

# feeding the X_test to model


## Single funtion for whole task

In [38]:
output_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output" 
input_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\genrenew\bollypop\bp01.mp3"

In [39]:
import os
import librosa
import numpy as np
from spleeter.separator import Separator

def process_and_prepare_test(input_audio, test_output, sample_set_rate, target_shape = (150,150), threshold = 0.01):
    """
    1. Separates into 4 stems.
    2. Takes 'other.wav'.
    3. Chunks into 3-second Mel-Spectrograms.
    4. Returns X_Test (NumPy array).
    """
    # --- STEP 1: Spleeter Separation ---
    # Initializing for 4 stems (Vocals, Drums, Bass, Other)
    separator = Separator('spleeter:4stems')
    separator.separate_to_file(input_audio, test_output)
    
    # Locate the 'other' stem
    base_name = os.path.splitext(os.path.basename(input_audio))[0]
    other_stem_path = os.path.join(test_output, base_name, "other.wav")
    
    if not os.path.exists(other_stem_path):
        raise FileNotFoundError(f"Spleeter failed to create {other_stem_path}")

    # --- STEP 2: Load and Chunk ---

    try:
        audio_data, sample_rate = librosa.load(other_stem_path, sr = sample_set_rate)
    except Exception as e:
        print(f"Skipping corrupted files: {other_stem_path}: {e}")
        
    #define the duration of each chunk and overlap
    chunk_duration = 4
    overlap_duration = 2
    data = []

    #convert duration to sample
    chunk_samples = int(chunk_duration*sample_rate)
    stride = int((chunk_duration-overlap_duration)*sample_rate) ## movement(4-2) * sr -> 2 second movement
    
    # sliding window logic
    # Use range with stride to ensure 'start + chunk_samples' never exceeds len(y)
    for start in range(0, len(audio_data) - chunk_samples + 1, stride):
        end = start + chunk_samples
        chunk = audio_data[start:end]

        # QUALITY CHECK: Check if chunk has actual sound (RMS energy)
        rms = np.sqrt(np.mean(chunk**2))
        if rms > threshold: 

            # melspectrogram part, this is the matrix we are getting the 
            mel_spectrogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate, n_mels = 150) #calculating spectrogram by this
            mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref= np.max)

        # 2. APPLY NORMALIZATION (Min-Max Scaling)
            # This ensures the test data matches the 0-1 range of your training data
            mel_min = mel_spectrogram.min()
            mel_max = mel_spectrogram.max()
            if mel_max - mel_min > 0:
                mel_norm = (mel_spectrogram - mel_min) / (mel_max - mel_min)
            else:
                mel_norm = mel_spectrogram # Handle silence/constant signal

            # 3. Resize and Prepare for CNN
            # (150, 150) -> (150, 150, 1) for the Conv2D input
            mel_reshaped = np.expand_dims(mel_norm, axis=-1)
            mel_final = resize(mel_reshaped, target_shape).numpy()

            data.append(mel_final)
                
    return np.array(data)

In [40]:
# --- STEP 1: FORCE PATH AT THE TOP ---
# This ensures that even background processes see your clean FFmpeg
ffmpeg_bin = r'C:\FFmpeg\ffmpeg-8.0.1-full_build\bin'
os.environ["PATH"] = ffmpeg_bin + os.pathsep + os.environ["PATH"]

import tensorflow as tf
from tensorflow.keras import backend as K

# Clear the session to reset the Graph stack
K.clear_session()

X_test = process_and_prepare_test(input_audio= input_path, test_output= output_path, sample_set_rate= 22050)

INFO:tensorflow:Using config: {'_model_dir': 'pretrained_models\\4stems', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': gpu_options {
  per_process_gpu_memory_fraction: 0.7
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
Instructions for updating:
Use output_signature instead
Instructions for updating:
Use output_signature instead
INFO:tensorflow:Calling model_fn.
INFO:tensorfl

RuntimeError: Graph is finalized and cannot be modified.

## Function by Function

In [3]:
output_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output" 
input_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\genrenew\bollypop\bp01.mp3"

In [9]:
def spleeter_conversion_first(input_audio, test_output):
    # --- STEP 1: Spleeter Separation ---
    # Initializing for 4 stems (Vocals, Drums, Bass, Other)
    # separator = Separator('spleeter:4stems')
    # separator.separate_to_file(input_audio, test_output)

# We call Spleeter as a system command. This avoids the "Finalized Graph" error.
    # It starts Spleeter, does the work, and closes it completely.

    python_exe = sys.executable
    cmd = f'"{python_exe}" -m spleeter separate -p spleeter:4stems -o "{test_output}" "{input_audio}"'
    subprocess.run(cmd, shell=True, check=True)
    
    # # Locate the 'other' stem
    # # base_name = os.path.splitext(os.path.basename(input_audio))[0]
    # # other_stem_path = os.path.join(test_output, base_name, "other.wav")
    
    # if not os.path.exists(other_stem_path):
    #     raise FileNotFoundError(f"Spleeter failed to create {other_stem_path}")

In [4]:
import os
from spleeter.separator import Separator

def spleeter_conversion(input_audio, test_output):   
    try:
        # 1. Initialize the separator
        # 'spleeter:4stems' extracts Vocals, Drums, Bass, and Other
        separator = Separator('spleeter:4stems')

        # 2. Perform the separation
        # This will create a folder inside test_output named after your audio file
        separator.separate_to_file(input_audio, test_output)
        
        print("✅ Separation Complete!")

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        # Common error: If your GPU memory is full, try restarting the kernel

In [5]:
# --- STEP 1: FORCE PATH AT THE TOP ---
# This ensures that even background processes see your clean FFmpeg
ffmpeg_bin = r'C:\ffmpeg\bin'
os.environ["PATH"] = ffmpeg_bin + os.pathsep + os.environ["PATH"]
spleeter_conversion(input_path, output_path)    

INFO:tensorflow:Using config: {'_model_dir': 'pretrained_models\\4stems', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': gpu_options {
  per_process_gpu_memory_fraction: 0.7
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
Instructions for updating:
Use output_signature instead
Instructions for updating:
Use output_signature instead
INFO:tensorflow:Calling model_fn.
INFO:tensorfl

In [12]:
other_stem_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output\bp01\other.wav"

In [9]:
#chunking part
def chunking(other_stem_path, sample_set_rate = 22050, target_shape = (150,150), threshold = 0.01 ):
        # --- STEP 2: Load and Chunk ---

    try:
        audio_data, sample_rate = librosa.load(other_stem_path, sr = sample_set_rate)
    except Exception as e:
        print(f"Skipping corrupted files: {other_stem_path}: {e}")
        
    #define the duration of each chunk and overlap
    chunk_duration = 4
    overlap_duration = 2
    data = []

    #convert duration to sample
    chunk_samples = int(chunk_duration*sample_rate)
    stride = int((chunk_duration-overlap_duration)*sample_rate) ## movement(4-2) * sr -> 2 second movement
    
    # sliding window logic
    # Use range with stride to ensure 'start + chunk_samples' never exceeds len(y)
    for start in range(0, len(audio_data) - chunk_samples + 1, stride):
        end = start + chunk_samples
        chunk = audio_data[start:end]

        # QUALITY CHECK: Check if chunk has actual sound (RMS energy)
        rms = np.sqrt(np.mean(chunk**2))
        if rms > threshold: 

            # melspectrogram part, this is the matrix we are getting the 
            mel_spectrogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate, n_mels = 150) #calculating spectrogram by this
            mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref= np.max)


            # 2. APPLY NORMALIZATION (Min-Max Scaling)
            # This ensures the test data matches the 0-1 range of your training data
            mel_min = mel_spectrogram.min()
            mel_max = mel_spectrogram.max()
            if mel_max - mel_min > 0:
                mel_norm = (mel_spectrogram - mel_min) / (mel_max - mel_min)
            else:
                mel_norm = mel_spectrogram # Handle silence/constant signal

            mel_spectrogram = resize(np.expand_dims(mel_norm, axis = -1), target_shape).numpy()

            # appending the data to the lists
            data.append(mel_spectrogram)
                
    return np.array(data)
    

In [10]:
X_test = chunking(other_stem_path=other_stem_path)

Skipping corrupted files: C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output\bp01\other.wav: name 'librosa' is not defined


UnboundLocalError: local variable 'sample_rate' referenced before assignment

In [12]:
X_test.shape

(21, 150, 150, 1)

In [13]:
X_test

array([[[[0.39576074],
         [0.39962536],
         [0.3848373 ],
         ...,
         [0.40010127],
         [0.44733778],
         [0.50516284]],

        [[0.3919411 ],
         [0.45674497],
         [0.47452354],
         ...,
         [0.4110363 ],
         [0.4289969 ],
         [0.45541677]],

        [[0.31387395],
         [0.45321652],
         [0.4927006 ],
         ...,
         [0.42603412],
         [0.46178952],
         [0.498403  ]],

        ...,

        [[0.00396991],
         [0.        ],
         [0.        ],
         ...,
         [0.        ],
         [0.07065872],
         [0.2084    ]],

        [[0.        ],
         [0.        ],
         [0.        ],
         ...,
         [0.        ],
         [0.06876469],
         [0.20621946]],

        [[0.        ],
         [0.        ],
         [0.        ],
         ...,
         [0.        ],
         [0.06903602],
         [0.20660886]]],


       [[[0.40208647],
         [0.4392682 ],
         [0.43

In [14]:
#predict function
def model_prediction(X_test, classes):
    y_pred = model.predict(X_test)
    predicted_categories = np.argmax(y_pred , axis = 1)
    unique_elements, counts = np.unique(predicted_categories, return_counts = True)
    max_element = np.max(counts)
    index = unique_elements[counts == max_element][0]
    return classes[index]

In [15]:
classes = ['bollypop', 'carnatic', 'ghazal', 'semiclassical', 'sufi']

In [16]:
result = model_prediction(X_test, classes=classes)
result

1/1 [==============================] - 6s 6s/step


'bollypop'